# YOLOv8 Object Detection — Tutorial

This notebook walks through the detection engine that powers the Streamlit app (`src/detector.py`) programmatically, outside the UI. It covers:

1. Loading a model and running single-image inference
2. Visualizing results
3. Batch inference
4. Comparing multiple model sizes
5. Video / webcam streaming with object tracking
6. A quick-start guide to fine-tuning YOLOv8 on a custom dataset

> Run this notebook from the repository root (`jupyter notebook notebooks/tutorial.ipynb`) so the `src` package resolves correctly.

In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path when launched from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.detector import YOLODetector, InferenceConfig, compare_models, get_device_info
from src.utils import draw_detections, pil_to_bgr, bgr_to_pil

print("Device:", get_device_info())

## 1. Load a model and run single-image inference

In [ ]:
detector = YOLODetector("yolov8n.pt", device="auto")

sample_path = config.SAMPLE_IMAGES_DIR / "bus.jpg"
image = Image.open(sample_path)

result = detector.predict_image(image, InferenceConfig(confidence=0.3))

print(f"Detections: {result.num_detections}")
print(f"Inference time: {result.inference_time_ms:.2f} ms")
print(f"Classes: {result.class_distribution}")

## 2. Visualize the annotated result

In [ ]:
bgr = pil_to_bgr(image)
annotated = draw_detections(
    bgr, result.boxes, result.class_ids, result.confidences, result.class_names
)

plt.figure(figsize=(10, 8))
plt.imshow(bgr_to_pil(annotated))
plt.axis("off")
plt.title(f"{result.num_detections} objects detected")
plt.show()

## 3. Batch inference

Passing a list of images to `predict_batch` runs a single forward pass over the whole batch — more efficient than looping `predict_image` when you have many images ready at once.

In [ ]:
image_paths = list(config.SAMPLE_IMAGES_DIR.glob("*.jpg"))
images = [Image.open(p) for p in image_paths]

batch_results = detector.predict_batch(images, InferenceConfig(confidence=0.25))

for path, res in zip(image_paths, batch_results):
    print(f"{path.name}: {res.num_detections} objects in {res.inference_time_ms:.1f} ms")

## 4. Compare model sizes

The same helper backs the Streamlit "Model Comparison" tab: run multiple YOLOv8 variants on one image and compare speed vs. detection count. Larger models download automatically the first time they're used (may take a minute).

In [ ]:
models_to_compare = ["yolov8n.pt", "yolov8s.pt"]  # add "yolov8m.pt" etc. if you have time/bandwidth

comparison = compare_models(image, models_to_compare, InferenceConfig(confidence=0.25))

for name, res in comparison.items():
    bench = config.MODEL_BENCHMARKS.get(name, {})
    print(
        f"{name:12s} | detections={res.num_detections:2d} | "
        f"time={res.inference_time_ms:6.1f} ms | "
        f"params={bench.get('params_m')}M | mAP50-95={bench.get('map50_95')}"
    )

## 5. Video / webcam streaming with tracking

`predict_video_stream` yields `(frame, DetectionResult)` pairs lazily, so memory stays flat regardless of video length. Set `use_tracking=True` to get persistent object IDs across frames via ByteTrack.

This cell is commented out by default since it needs a real video file or webcam — uncomment to try it locally.

In [ ]:
# video_path = config.SAMPLE_VIDEO_DIR / "your_video.mp4"
# frame_count = 0
# for frame, res in detector.predict_video_stream(str(video_path), use_tracking=True):
#     frame_count += 1
#     if frame_count % 30 == 0:
#         print(f"Frame {frame_count}: {res.num_detections} objects, ids={res.track_ids}")
#     if frame_count >= 150:
#         break

## 6. Fine-tuning YOLOv8 on a custom dataset

The app ships with pretrained COCO weights, but Ultralytics makes custom training a few lines of code. This is a reference walkthrough — it expects a dataset in YOLO format (images + `.txt` label files) described by a `data.yaml`.

```
my_dataset/
├── data.yaml
├── train/images, train/labels
└── val/images,   val/labels
```

`data.yaml` example:
```yaml
path: my_dataset
train: train/images
val: val/images
names:
  0: my_class_a
  1: my_class_b
```

In [ ]:
# from ultralytics import YOLO
#
# model = YOLO("yolov8n.pt")  # start from pretrained COCO weights
# training_results = model.train(
#     data="my_dataset/data.yaml",
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     device="auto",
#     project="runs/train",
#     name="custom_yolov8n",
# )
#
# # Evaluate on the validation split
# metrics = model.val()
# print(metrics.box.map)  # mAP50-95
#
# # Point src/config.py's AVAILABLE_MODELS at the resulting weights to use
# # your fine-tuned model in the Streamlit app:
# #   "My Custom Model": "runs/train/custom_yolov8n/weights/best.pt"

## Summary

- `YOLODetector` wraps model loading, single/batch/video inference, and tracking behind one small API.
- `ModelCache` (used internally by `compare_models` and the Streamlit app) avoids reloading weights across calls.
- The exact same `detector.py` module powers the web app, the REST API (`src/api.py`), and this notebook — no duplicated logic.

Next steps: open `docs/USAGE.md` for CLI examples, or `streamlit run src/app.py` to explore the full UI.